#only use this on the jupyterhub first time use

#this is the training part

In [3]:
from ultralytics import YOLO
import yaml
from datetime import datetime
import shutil
import os
import pathlib
import torch
import gc

In [6]:


with open("settings.yaml", "r") as f:
    settings = yaml.safe_load(f)

training_datasets = settings["training_datasets"]
training_configurations = settings["training_configurations"]
model_paths = ["yolo11n.pt", "yolo11x.pt"]  

start_time = datetime.now()

for dataset in training_datasets:
    dataset_yaml = pathlib.Path.cwd().parent / "datasets" / dataset["type"] / dataset["name"] / "dataset.yaml"
    for configuration in training_configurations:
        for model_path in model_paths:
            model = YOLO(model_path)
            model_name = pathlib.Path(model.ckpt_path).stem
            timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M")
            run_name = f"{timestamp}_{dataset['name']}"

            train_kwargs = {
                "data": str(dataset_yaml),
                "name": run_name,
                **{k: configuration[k] for k in ["time", "imgsz", "patience", "epochs", "multi_scale", "batch"] if k in configuration}
            }

            train_results = model.train(**train_kwargs)

            # Move results directory
            for subdir in ["detect", "segment"]:
                source_dir = pathlib.Path("runs") / subdir / run_name
                if source_dir.is_dir():
                    target_dir = pathlib.Path.cwd().parent / "models" / dataset["type"] / run_name
                    shutil.move(str(source_dir), str(target_dir))
                    print(f"Model saved to: {target_dir}")
                    break
            else:
                print("No valid run directory found")

            # Print training info
            end_time = datetime.now()
            duration = end_time - start_time
            hours, minutes = divmod(int(duration.total_seconds() // 60), 60)
            print(f"Training of {dataset['name']} with {model_name} completed")
            print(f"Training duration: {hours}h {minutes}m")

            # Release memory
            del train_results
            del model
            torch.cuda.empty_cache()

New https://pypi.org/project/ultralytics/8.3.192 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.190 🚀 Python-3.10.18 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A4000, 15985MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=0.9, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=Fal

KeyboardInterrupt: 

In [ ]:
!nvidia-smi